In [1]:
#0.确认当前位置
import os
os.getcwd()

'/root/project/RAG_Agent/docs'

In [3]:
# 1. 读取json
import json
from pathlib import Path

json_path = Path("../data/mineru_out/2/auto/2_content_list.json")

with open(json_path, "r", encoding="utf-8") as f:
    content_list = json.load(f)

type(content_list), len(content_list)

(list, 134)

In [4]:
# 2. 找到表格
table_indices = []
for i,item in enumerate(content_list):
    if isinstance(item, dict) and item.get("type") == "table":
        table_indices.append(i)
        print(f"index={i}, page_idx = {item.get('page_idx')}, captioin={item.get('table_id')}")
print("table 总数，", len(table_indices))

index=10, page_idx = 0, captioin=None
index=11, page_idx = 0, captioin=None
index=12, page_idx = 0, captioin=None
index=40, page_idx = 0, captioin=None
index=41, page_idx = 1, captioin=None
index=42, page_idx = 1, captioin=None
index=47, page_idx = 2, captioin=None
index=48, page_idx = 2, captioin=None
index=49, page_idx = 2, captioin=None
index=50, page_idx = 2, captioin=None
table 总数， 10


In [5]:
# 3. 将table_body 从HTML解析为可读的二维表df
import pandas as pd
from io import StringIO

target_idx = 10
table_block = content_list[target_idx]

table_html = table_block["table_body"]
df = pd.read_html(StringIO(table_html))[0]
df


,0,1,2,3,4
0,(%),今年 至今,1 个月,3 个月,12 个月
1,绝对,49.9,14.9,53.1,23.1
2,相对深圳成指,51.5,21.1,53.1,(4.7)


In [6]:
# 4.将df转变为大模型易读的结构化文本
def df_to_llm_text(df):
    lines = []
    
    # strip()的作用是去掉字符串开头和结尾的空白字符
    header = [str(x).strip() for x in df.iloc[0].tolist()]
    lines.append("表头: " + " | ".join(header))
    
    
    for i in range(1, len(df)):
        row = [str(x).strip() for x in df.iloc[i].tolist()]
        row_text = " | ".join(row)
        lines.append(f"第{i}行: {row_text}")
    
    return "\n".join(lines)

llm_text = df_to_llm_text(df)
print(llm_text)

表头: (%) | 今年 至今 | 1 个月 | 3 个月 | 12 个月
第1行: 绝对 | 49.9 | 14.9 | 53.1 | 23.1
第2行: 相对深圳成指 | 51.5 | 21.1 | 53.1 | (4.7)


In [7]:
for idx in table_indices:
    table_block = content_list[idx]
    table_html = table_block["table_body"]
    
    df = pd.read_html(StringIO(table_html))[0]
    llm_text = df_to_llm_text(df)
    
    content_list[idx]["table_body"] = llm_text

print(f"已批量转换 {len(table_indices)} 个 table 的 table_body")
print(content_list[table_indices[0]]["table_body"][:300])

已批量转换 10 个 table 的 table_body
表头: (%) | 今年 至今 | 1 个月 | 3 个月 | 12 个月
第1行: 绝对 | 49.9 | 14.9 | 53.1 | 23.1
第2行: 相对深圳成指 | 51.5 | 21.1 | 53.1 | (4.7)


In [8]:
from pathlib import Path
import json

output_path = Path("../data/mineru_out/2/auto/2_content_list_table_llm.json")
output_path.parent.mkdir(parents=True, exist_ok=True)
# 打开一个文件 并把这个文件对象命名为f
with open(output_path, "w", encoding="utf-8") as f:
    #json.dump(...)是把Python里的对象写进一个json文件里
    json.dump(content_list, f, ensure_ascii=False, indent=2)

print(f"已保存到: {output_path}")

已保存到: ../data/mineru_out/2/auto/2_content_list_table_llm.json


In [10]:
for idx in table_indices:
    print("=" * 80)
    print(f"table index: {idx}")
    print("caption:", content_list[idx].get("table_caption"))
    print("新的 table_body 预览:")
    print(content_list[idx]["table_body"][:500])   # 只看前500个字符
    print()

table index: 10
caption: []
新的 table_body 预览:
表头: (%) | 今年 至今 | 1 个月 | 3 个月 | 12 个月
第1行: 绝对 | 49.9 | 14.9 | 53.1 | 23.1
第2行: 相对深圳成指 | 51.5 | 21.1 | 53.1 | (4.7)

table index: 11
caption: []
新的 table_body 预览:
表头: 发行股数 (百万) | 3,368.65 3,366.48
第1行: 流通股 (百万) 总市值 (人民币 百万) | 89673.35
第2行: 3 个月日均交易额 (人民币 百万) | 1866.55
第3行: 主要股东 | nan

table index: 12
caption: []
新的 table_body 预览:
表头: 浙江卫星控股股份有限公司(%) 34.6

table index: 40
caption: ['[Table_Fin投资摘要']
新的 table_body 预览:
表头: 年结日：12月31日 | 2024 | 2025 | 2026E | 2027E | 2028E
第1行: 主营收入(人民币百万) | 45648 | 46068 | 58632 | 64369 | 65715
第2行: 增长率(%) | 10.0 | 0.9 | 27.3 | 9.8 | 2.1
第3行: EBITDA(人民币百万) | 12261 | 12776 | 15021 | 17119 | 18048
第4行: 归母净利润(人民币百万) | 6072 | 5311 | 7952 | 9355 | 9740
第5行: 增长率(%) | 26.8 | (12.5) | 49.7 | 17.7 | 4.1
第6行: 最新股本摊薄每股收益(人民币) | 1.80 | 1.58 | 2.36 | 2.78 | 2.89
第7行: 原预测归母净利润(人民币 百万) | / | / | 7827 | 9328 | /
第8行: 调整幅度(%) | / | / | 1.60 | 0.29 | /
第9行: 市盈率(倍) | 14.8 | 16.9 | 11.3 | 9.6 | 9.

table index: 41
caption: ['图表 1.公